In [1]:
from pathlib import Path

import whisper
import librosa
import numpy as np
import pandas as pd

In [2]:
DATASET_PATH = Path("../data/raw/librispeech/LibriSpeech/test-clean")

audio_files = list(DATASET_PATH.rglob("*.flac"))

print("Dataset exists:", DATASET_PATH.exists())
print("Total audio files:", len(audio_files))
print("First file:", audio_files[0])

Dataset exists: True
Total audio files: 2620
First file: ..\data\raw\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0000.flac


In [3]:
import whisper

whisper_model = whisper.load_model("base")

print("Whisper model loaded.")

Whisper model loaded.


In [4]:
def get_transcript_for_audio(audio_path):
    speaker_id = audio_path.parent.parent.name
    chapter_id = audio_path.parent.name

    transcript_file = audio_path.parent / f"{speaker_id}-{chapter_id}.trans.txt"

    with open(transcript_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    audio_id = audio_path.stem

    for line in lines:
        if line.startswith(audio_id):
            return line.replace(audio_id, "").strip()

    return None

In [5]:
sample_audio = audio_files[0]
sample_transcript = get_transcript_for_audio(sample_audio)

print("Audio:", sample_audio)
print("Transcript:", sample_transcript)

Audio: ..\data\raw\librispeech\LibriSpeech\test-clean\1089\134686\1089-134686-0000.flac
Transcript: HE HOPED THERE WOULD BE STEW FOR DINNER TURNIPS AND CARROTS AND BRUISED POTATOES AND FAT MUTTON PIECES TO BE LADLED OUT IN THICK PEPPERED FLOUR FATTENED SAUCE


In [6]:
def extract_audio_features(audio_path, transcript):
    audio, sr = librosa.load(audio_path, sr=16000)

    duration = len(audio) / sr
    word_count = len(transcript.split())
    wpm = word_count / (duration / 60)

    rms = librosa.feature.rms(y=audio)
    zcr = librosa.feature.zero_crossing_rate(audio)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    features = {
        "audio_file": str(audio_path),
        "transcript": transcript,
        "duration_seconds": round(duration, 2),
        "word_count": word_count,
        "words_per_minute": round(wpm, 2),
        "average_rms_energy": round(float(np.mean(rms)), 6),
        "average_zero_crossing_rate": round(float(np.mean(zcr)), 6),
        "spectral_centroid": round(float(np.mean(spectral_centroid)), 2),
        "spectral_bandwidth": round(float(np.mean(spectral_bandwidth)), 2)
    }

    mfcc_mean = np.mean(mfcc, axis=1)
    for i, value in enumerate(mfcc_mean):
        features[f"mfcc_{i+1}"] = round(float(value), 4)

    return features

In [7]:
sample_features = extract_audio_features(sample_audio, sample_transcript)

pd.DataFrame([sample_features]).T

,0
audio_file,..\data\raw\librispeech\LibriSpeech\test-clean...
transcript,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...
duration_seconds,10.44
word_count,28
words_per_minute,161.0
average_rms_energy,0.034645
average_zero_crossing_rate,0.127497
spectral_centroid,1704.13
spectral_bandwidth,1476.58
mfcc_1,-300.5381


In [8]:
dataset = []

for audio_path in audio_files[:100]:

    transcript = get_transcript_for_audio(audio_path)

    if transcript is None:
        continue

    features = extract_audio_features(audio_path, transcript)

    dataset.append(features)

print("Processed", len(dataset), "audio files.")

Processed 100 audio files.


In [9]:
dataset_df = pd.DataFrame(dataset)

print(dataset_df.shape)

dataset_df.head()

(100, 22)


,audio_file,transcript,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,...,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13
0,..\data\raw\librispeech\LibriSpeech\test-clean...,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,10.44,28,161.00,0.034645,0.127497,1704.13,1476.58,-300.5381,...,34.5156,3.6980,3.7405,-7.7800,-5.0308,-1.1619,-0.9489,-0.2123,1.6172,-5.0724
1,..\data\raw\librispeech\LibriSpeech\test-clean...,STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,3.27,8,146.56,0.019861,0.107820,1741.59,1628.54,-364.2475,...,40.9727,3.8557,11.0615,-3.9844,-0.6017,4.0342,-2.4162,4.9232,1.5388,1.1172
2,..\data\raw\librispeech\LibriSpeech\test-clean...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,6.62,18,163.02,0.038250,0.087339,1390.03,1353.31,-303.5464,...,26.2034,-7.2205,2.7213,-7.6972,-8.8562,1.2397,-1.2306,0.2887,-0.0212,-3.2952
3,..\data\raw\librispeech\LibriSpeech\test-clean...,HELLO BERTIE ANY GOOD IN YOUR MIND,2.68,7,156.72,0.063083,0.065348,1312.21,1378.24,-313.2429,...,21.1822,-5.3696,-9.6886,-9.5899,-7.0875,-10.2026,-2.0147,-4.1345,-0.2080,-6.3018
4,..\data\raw\librispeech\LibriSpeech\test-clean...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,5.22,11,126.56,0.048197,0.096332,1548.08,1483.26,-298.8683,...,29.3365,2.9664,-3.6720,-11.2932,-2.4028,-1.2842,0.8973,-3.4142,1.5993,-3.0086


In [10]:
dataset_df.to_csv("../data/processed/fluentiq_dataset_v1.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


In [11]:
loaded_df = pd.read_csv("../data/processed/fluentiq_dataset_v1.csv")

loaded_df.head()

,audio_file,transcript,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,...,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13
0,..\data\raw\librispeech\LibriSpeech\test-clean...,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,10.44,28,161.00,0.034645,0.127497,1704.13,1476.58,-300.5381,...,34.5156,3.6980,3.7405,-7.7800,-5.0308,-1.1619,-0.9489,-0.2123,1.6172,-5.0724
1,..\data\raw\librispeech\LibriSpeech\test-clean...,STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,3.27,8,146.56,0.019861,0.107820,1741.59,1628.54,-364.2475,...,40.9727,3.8557,11.0615,-3.9844,-0.6017,4.0342,-2.4162,4.9232,1.5388,1.1172
2,..\data\raw\librispeech\LibriSpeech\test-clean...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,6.62,18,163.02,0.038250,0.087339,1390.03,1353.31,-303.5464,...,26.2034,-7.2205,2.7213,-7.6972,-8.8562,1.2397,-1.2306,0.2887,-0.0212,-3.2952
3,..\data\raw\librispeech\LibriSpeech\test-clean...,HELLO BERTIE ANY GOOD IN YOUR MIND,2.68,7,156.72,0.063083,0.065348,1312.21,1378.24,-313.2429,...,21.1822,-5.3696,-9.6886,-9.5899,-7.0875,-10.2026,-2.0147,-4.1345,-0.2080,-6.3018
4,..\data\raw\librispeech\LibriSpeech\test-clean...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,5.22,11,126.56,0.048197,0.096332,1548.08,1483.26,-298.8683,...,29.3365,2.9664,-3.6720,-11.2932,-2.4028,-1.2842,0.8973,-3.4142,1.5993,-3.0086


In [12]:
def calculate_fluentiq_scores(row):
    wpm = row["words_per_minute"]
    rms = row["average_rms_energy"]
    zcr = row["average_zero_crossing_rate"]
    word_count = row["word_count"]

    # Fluency score based on speaking pace
    if 120 <= wpm <= 160:
        fluency_score = 90
    elif 100 <= wpm < 120 or 160 < wpm <= 180:
        fluency_score = 75
    else:
        fluency_score = 60

    # Confidence score based on voice energy
    confidence_score = min(100, max(40, rms * 2000))

    # Clarity score based on ZCR
    clarity_score = min(100, max(40, (1 - zcr) * 100))

    # Vocabulary score based on word count
    vocabulary_score = min(100, max(50, word_count * 3))

    overall_score = (
        fluency_score * 0.35 +
        confidence_score * 0.25 +
        clarity_score * 0.20 +
        vocabulary_score * 0.20
    )

    if overall_score >= 80:
        level = "Advanced"
    elif overall_score >= 65:
        level = "Intermediate"
    else:
        level = "Beginner"

    return pd.Series({
        "fluency_score": round(fluency_score, 2),
        "confidence_score": round(confidence_score, 2),
        "clarity_score": round(clarity_score, 2),
        "vocabulary_score": round(vocabulary_score, 2),
        "overall_score": round(overall_score, 2),
        "communication_level": level
    })

In [13]:
score_df = dataset_df.apply(calculate_fluentiq_scores, axis=1)

final_df = pd.concat([dataset_df, score_df], axis=1)

final_df.head()

,audio_file,transcript,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,...,mfcc_10,mfcc_11,mfcc_12,mfcc_13,fluency_score,confidence_score,clarity_score,vocabulary_score,overall_score,communication_level
0,..\data\raw\librispeech\LibriSpeech\test-clean...,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,10.44,28,161.00,0.034645,0.127497,1704.13,1476.58,-300.5381,...,-0.9489,-0.2123,1.6172,-5.0724,75,69.29,87.25,84,77.82,Intermediate
1,..\data\raw\librispeech\LibriSpeech\test-clean...,STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,3.27,8,146.56,0.019861,0.107820,1741.59,1628.54,-364.2475,...,-2.4162,4.9232,1.5388,1.1172,90,40.00,89.22,50,69.34,Intermediate
2,..\data\raw\librispeech\LibriSpeech\test-clean...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,6.62,18,163.02,0.038250,0.087339,1390.03,1353.31,-303.5464,...,-1.2306,0.2887,-0.0212,-3.2952,75,76.50,91.27,54,74.43,Intermediate
3,..\data\raw\librispeech\LibriSpeech\test-clean...,HELLO BERTIE ANY GOOD IN YOUR MIND,2.68,7,156.72,0.063083,0.065348,1312.21,1378.24,-313.2429,...,-2.0147,-4.1345,-0.2080,-6.3018,90,100.00,93.47,50,85.19,Advanced
4,..\data\raw\librispeech\LibriSpeech\test-clean...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,5.22,11,126.56,0.048197,0.096332,1548.08,1483.26,-298.8683,...,0.8973,-3.4142,1.5993,-3.0086,90,96.39,90.37,50,83.67,Advanced


In [14]:
final_df.to_csv("../data/processed/fluentiq_scored_dataset_v1.csv", index=False)

print("Final scored dataset saved successfully.")
print(final_df.shape)

Final scored dataset saved successfully.
(100, 28)


In [15]:
final_df["communication_level"].value_counts()

communication_level
Intermediate    53
Advanced        40
Beginner         7
Name: count, dtype: int64